# Prime Numbers Lab — Notebook 17: Higher-Order Transition Memory and Shuffle Baseline

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** test whether residue-class transition structure contains measurable second-order memory beyond a first-order transition operator.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

Notebook 16 built a first-order transition operator. Notebook 17 compares empirical two-step transitions against the Markov prediction \(P^2\), then tests that residual against shuffled baselines.

## 0. Setup

This notebook follows the established `prime-numbers-lab` template:

1. define one constraint  
2. generate one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and TeX  
6. package results into a root-level export zip

In [ ]:
# Standard library
from pathlib import Path
import json
import math
import zipfile

# Data / compute
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Notebook identity
NOTEBOOK_ID = "17_higher_order_transition_memory_shuffle_baseline"
NOTEBOOK_TITLE = "Higher-Order Transition Memory and Shuffle Baseline"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

# Output directories
OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")

## 1. Premise

Notebook 16 showed that prime residue transitions mod 30 form a finite transition operator. Notebook 17 asks whether two-step structure remains after the first-order prediction is accounted for.

\[
P_{ij}=\Pr(r_{n+1}=j\mid r_n=i)
\]

\[
Q_{ik}=\Pr(r_{n+2}=k\mid r_n=i)
\]

\[
\Delta^{(2)} = Q-P^2
\]

Core question:

> Does prime residue structure retain higher-order memory beyond a first-order transition operator?

## 2. Constraint definition

The first-order transition operator is:

\[
P_{ij}=rac{\#\{r_n=i, r_{n+1}=j\}}{\#\{r_n=i\}}
\]

The empirical two-step operator is:

\[
Q_{ik}=rac{\#\{r_n=i, r_{n+2}=k\}}{\#\{r_n=i\}}
\]

The first-order Markov prediction is:

\[
Q^{(1)}=P^2
\]

The higher-order residual is:

\[
\Delta^{(2)}_{ik}=Q_{ik}-(P^2)_{ik}
\]

In [ ]:
# Notebook-specific parameters
N_MAX = 2_000_000
RANDOM_SEED = 9423
RESIDUES30 = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
RESIDUE_LABELS = [str(r) for r in RESIDUES30]
WINDOW_COUNT = 14
MIN_WINDOW_TRIPLES = 250
SHUFFLE_COUNT = 200
TOP_N = 20
rng = np.random.default_rng(RANDOM_SEED)
params = dict(N_MAX=N_MAX, RANDOM_SEED=RANDOM_SEED, RESIDUES30=RESIDUES30.tolist(), WINDOW_COUNT=WINDOW_COUNT, MIN_WINDOW_TRIPLES=MIN_WINDOW_TRIPLES, SHUFFLE_COUNT=SHUFFLE_COUNT, TOP_N=TOP_N, NOTEBOOK_ID=NOTEBOOK_ID, NOTEBOOK_TITLE=NOTEBOOK_TITLE)
params

## 3. Data generation

Generate primes up to \(N_{max}\), remove \(2,3,5\), compute residue sequences, first-order transitions, and two-step triples.

In [ ]:
def generate_primes(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.array([], dtype=int)
    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False
    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max+1:i] = False
    return np.nonzero(sieve)[0].astype(int)

primes_all = generate_primes(N_MAX)
primes = primes_all[primes_all > 5]
residues = primes % 30
valid = np.isin(residues, RESIDUES30)
primes = primes[valid]
residues = residues[valid]

r0, r1, r2 = residues[:-2], residues[1:-1], residues[2:]
anchor_x = primes[:-2]
first_gaps = primes[1:-1] - primes[:-2]
second_gaps = primes[2:] - primes[1:-1]
combined_gap = primes[2:] - primes[:-2]

summary = {
    "n_max": int(N_MAX),
    "prime_count_excluding_2_3_5": int(len(primes)),
    "first_order_transition_count": int(len(residues)-1),
    "two_step_triple_count": int(len(r0)),
    "mean_first_gap": float(np.mean(first_gaps)),
    "mean_second_gap": float(np.mean(second_gaps)),
    "mean_combined_gap": float(np.mean(combined_gap)),
}
summary

## 4. First-order and two-step operators

In [ ]:
residue_to_index = {int(r): i for i, r in enumerate(RESIDUES30)}
n_res = len(RESIDUES30)

def row_normalize(counts: np.ndarray) -> np.ndarray:
    row_sums = counts.sum(axis=1, keepdims=True)
    return np.divide(counts, row_sums, out=np.zeros_like(counts, dtype=float), where=row_sums > 0)

def compute_operators(seq: np.ndarray):
    first = np.zeros((n_res, n_res), dtype=float)
    second = np.zeros((n_res, n_res), dtype=float)
    for a,b in zip(seq[:-1], seq[1:]):
        first[residue_to_index[int(a)], residue_to_index[int(b)]] += 1
    for a,c in zip(seq[:-2], seq[2:]):
        second[residue_to_index[int(a)], residue_to_index[int(c)]] += 1
    P_local = row_normalize(first)
    Q_local = row_normalize(second)
    D_local = Q_local - P_local @ P_local
    return first, second, P_local, Q_local, D_local

first_counts, second_counts, P, Q_emp, Delta2 = compute_operators(residues)
Q_markov = P @ P
Delta2 = Q_emp - Q_markov
P_df = pd.DataFrame(P, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
Q_emp_df = pd.DataFrame(Q_emp, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
Q_markov_df = pd.DataFrame(Q_markov, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
Delta2_df = pd.DataFrame(Delta2, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
P_df.round(4), Q_emp_df.round(4), Delta2_df.round(5)

## 5. Higher-order residual metrics

In [ ]:
def matrix_l1(A): return float(np.mean(np.abs(A)))
def matrix_l2(A): return float(np.sqrt(np.mean(A**2)))
def normalized_entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p*np.log(p))/np.log(n_res)) if len(p) else 0.0

higher_order_l1 = matrix_l1(Delta2)
higher_order_l2 = matrix_l2(Delta2)
higher_order_max_abs = float(np.max(np.abs(Delta2)))
row_entropy_P = np.array([normalized_entropy(P[i]) for i in range(n_res)])
row_entropy_Q = np.array([normalized_entropy(Q_emp[i]) for i in range(n_res)])
row_entropy_delta = row_entropy_Q - row_entropy_P
operator_metrics_df = pd.DataFrame({"metric":["higher_order_l1","higher_order_l2","higher_order_max_abs","mean_first_order_entropy","mean_two_step_entropy","mean_entropy_delta"], "value":[higher_order_l1,higher_order_l2,higher_order_max_abs,float(row_entropy_P.mean()),float(row_entropy_Q.mean()),float(row_entropy_delta.mean())]})
records=[]
for i,a in enumerate(RESIDUES30):
    for k,c in enumerate(RESIDUES30):
        records.append({"two_step_transition":f"{a}->{c}","anchor_residue":int(a),"two_step_residue":int(c),"empirical_two_step_probability":float(Q_emp[i,k]),"markov_predicted_probability":float(Q_markov[i,k]),"delta":float(Delta2[i,k]),"abs_delta":float(abs(Delta2[i,k]))})
two_step_delta_df = pd.DataFrame(records).sort_values("abs_delta", ascending=False)
entropy_comparison_df = pd.DataFrame({"anchor_residue":RESIDUES30,"first_order_entropy":row_entropy_P,"two_step_entropy":row_entropy_Q,"entropy_delta":row_entropy_delta})
operator_metrics_df, two_step_delta_df.head(TOP_N), entropy_comparison_df

## 6. Shuffle baseline

Destroy order while preserving residue counts, then recompute \(\|Q_s-P_s^2\|\).

In [ ]:
def residual_metrics_for_sequence(seq):
    _, _, P_s, Q_s, D_s = compute_operators(seq)
    return matrix_l1(D_s), matrix_l2(D_s), float(np.max(np.abs(D_s)))

shuffle_rows=[]
for s in range(SHUFFLE_COUNT):
    shuffled = residues.copy()
    rng.shuffle(shuffled)
    l1,l2,mx = residual_metrics_for_sequence(shuffled)
    shuffle_rows.append({"shuffle_index":s,"shuffle_l1":l1,"shuffle_l2":l2,"shuffle_max_abs":mx})
shuffle_df = pd.DataFrame(shuffle_rows)
shuffle_summary = {
    "real_l1": higher_order_l1,
    "real_l2": higher_order_l2,
    "real_max_abs": higher_order_max_abs,
    "shuffle_mean_l1": float(shuffle_df.shuffle_l1.mean()),
    "shuffle_std_l1": float(shuffle_df.shuffle_l1.std()),
    "shuffle_mean_l2": float(shuffle_df.shuffle_l2.mean()),
    "shuffle_std_l2": float(shuffle_df.shuffle_l2.std()),
    "shuffle_mean_max_abs": float(shuffle_df.shuffle_max_abs.mean()),
    "shuffle_std_max_abs": float(shuffle_df.shuffle_max_abs.std()),
}
shuffle_summary["l1_excess_over_shuffle"] = float(higher_order_l1 - shuffle_summary["shuffle_mean_l1"])
shuffle_summary["l2_excess_over_shuffle"] = float(higher_order_l2 - shuffle_summary["shuffle_mean_l2"])
shuffle_summary_df = pd.DataFrame([shuffle_summary])
shuffle_summary_df

## 7. Conditional chain residuals

Measure:

\[
P(k\mid i,j)-P(k\mid j)
\]

In [ ]:
chain_counts, pair_counts = {}, {}
for a,b,c in zip(r0,r1,r2):
    pair_counts[(int(a),int(b))] = pair_counts.get((int(a),int(b)),0)+1
    chain_counts[(int(a),int(b),int(c))] = chain_counts.get((int(a),int(b),int(c)),0)+1
conditional_rows=[]
for (a,b,c), count in chain_counts.items():
    if pair_counts[(a,b)] < 25: continue
    empirical = count / pair_counts[(a,b)]
    baseline = P[residue_to_index[b], residue_to_index[c]]
    delta = empirical - baseline
    conditional_rows.append({"chain":f"{a}->{b}->{c}","anchor_residue":a,"middle_residue":b,"next_residue":c,"count":int(count),"pair_count":int(pair_counts[(a,b)]),"empirical_p_next_given_pair":float(empirical),"first_order_baseline":float(baseline),"conditional_delta":float(delta),"abs_conditional_delta":float(abs(delta))})
conditional_chain_df = pd.DataFrame(conditional_rows).sort_values("abs_conditional_delta", ascending=False)
conditional_chain_df.head(TOP_N)

## 8. Windowed higher-order structure

In [ ]:
raw_edges = np.unique(np.logspace(np.log10(int(anchor_x.min())), np.log10(int(anchor_x.max())), WINDOW_COUNT+1).astype(int))
raw_edges[0] = int(anchor_x.min()); raw_edges[-1] = int(anchor_x.max())
window_rows=[]; window_delta_rows=[]
for idx,(left,right) in enumerate(zip(raw_edges[:-1], raw_edges[1:]), start=1):
    seq_window = residues[(primes >= left) & (primes < right)]
    if len(seq_window) < MIN_WINDOW_TRIPLES: continue
    _, _, P_w, Q_w, D_w = compute_operators(seq_window)
    window_rows.append({"window_index":idx,"left":int(left),"right":int(right),"midpoint":float(math.sqrt(left*right)),"triple_count":int(max(len(seq_window)-2,0)),"two_step_l1":matrix_l1(D_w),"two_step_l2":matrix_l2(D_w),"two_step_max_abs":float(np.max(np.abs(D_w)))})
    for i,a in enumerate(RESIDUES30):
        for k,c in enumerate(RESIDUES30):
            window_delta_rows.append({"window_index":idx,"midpoint":float(math.sqrt(left*right)),"two_step_transition":f"{a}->{c}","anchor_residue":int(a),"two_step_residue":int(c),"delta":float(D_w[i,k]),"abs_delta":float(abs(D_w[i,k]))})
window_higher_order_df = pd.DataFrame(window_rows)
window_delta_grid_df = pd.DataFrame(window_delta_rows)
window_higher_order_df

## 9. Visualization

Notebook 17 produces higher-order transition-memory figures.

In [ ]:
# Figure 1 — empirical two-step operator heatmap
fig, ax = plt.subplots(figsize=(9,7)); im=ax.imshow(Q_emp, aspect="auto"); ax.set_title("Empirical two-step residue operator mod30"); ax.set_xlabel("residue p_{n+2} mod 30"); ax.set_ylabel("anchor residue p_n mod 30"); ax.set_xticks(np.arange(n_res)); ax.set_yticks(np.arange(n_res)); ax.set_xticklabels(RESIDUE_LABELS); ax.set_yticklabels(RESIDUE_LABELS); fig.colorbar(im, ax=ax, label="empirical probability")
for i in range(n_res):
    for j in range(n_res): ax.text(j,i,f"{Q_emp[i,j]:.2f}",ha="center",va="center",fontsize=8)
fig1_path=FIG_DIR/f"{NOTEBOOK_NUM}_empirical_two_step_operator_heatmap.png"; fig.savefig(fig1_path,dpi=180,bbox_inches="tight"); plt.show(); fig1_path

In [ ]:
# Figure 2 — Markov-predicted two-step operator heatmap
fig, ax = plt.subplots(figsize=(9,7)); im=ax.imshow(Q_markov, aspect="auto"); ax.set_title("Markov-predicted two-step operator P²"); ax.set_xlabel("residue p_{n+2} mod 30"); ax.set_ylabel("anchor residue p_n mod 30"); ax.set_xticks(np.arange(n_res)); ax.set_yticks(np.arange(n_res)); ax.set_xticklabels(RESIDUE_LABELS); ax.set_yticklabels(RESIDUE_LABELS); fig.colorbar(im, ax=ax, label="predicted probability")
for i in range(n_res):
    for j in range(n_res): ax.text(j,i,f"{Q_markov[i,j]:.2f}",ha="center",va="center",fontsize=8)
fig2_path=FIG_DIR/f"{NOTEBOOK_NUM}_markov_predicted_two_step_operator_heatmap.png"; fig.savefig(fig2_path,dpi=180,bbox_inches="tight"); plt.show(); fig2_path

In [ ]:
# Figure 3 — two-step residual heatmap
fig, ax = plt.subplots(figsize=(9,7)); im=ax.imshow(Delta2, aspect="auto"); ax.set_title("Two-step residual operator: empirical minus P²"); ax.set_xlabel("residue p_{n+2} mod 30"); ax.set_ylabel("anchor residue p_n mod 30"); ax.set_xticks(np.arange(n_res)); ax.set_yticks(np.arange(n_res)); ax.set_xticklabels(RESIDUE_LABELS); ax.set_yticklabels(RESIDUE_LABELS); fig.colorbar(im, ax=ax, label="delta probability")
for i in range(n_res):
    for j in range(n_res): ax.text(j,i,f"{Delta2[i,j]:+.3f}",ha="center",va="center",fontsize=7)
fig3_path=FIG_DIR/f"{NOTEBOOK_NUM}_two_step_operator_delta_heatmap.png"; fig.savefig(fig3_path,dpi=180,bbox_inches="tight"); plt.show(); fig3_path

In [ ]:
# Figure 4 — top two-step residual transitions
top=two_step_delta_df.head(TOP_N).iloc[::-1]
fig, ax = plt.subplots(figsize=(10,7)); ax.barh(top["two_step_transition"], top["abs_delta"]); ax.set_title("Top two-step residual transitions"); ax.set_xlabel("absolute delta |Q - P²|"); ax.set_ylabel("two-step transition"); ax.grid(True,axis="x",alpha=0.3)
fig4_path=FIG_DIR/f"{NOTEBOOK_NUM}_top_two_step_residual_transitions.png"; fig.savefig(fig4_path,dpi=180,bbox_inches="tight"); plt.show(); fig4_path

In [ ]:
# Figure 5 — real vs shuffle L1 residual baseline
fig, ax = plt.subplots(figsize=(8,5)); ax.hist(shuffle_df["shuffle_l1"], bins=25, alpha=0.7, label="shuffle L1"); ax.axvline(higher_order_l1, linewidth=2, label="real L1"); ax.set_title("Real two-step residual L1 vs shuffle baseline"); ax.set_xlabel("two-step residual L1"); ax.set_ylabel("frequency"); ax.legend(); ax.grid(True,alpha=0.3)
fig5_path=FIG_DIR/f"{NOTEBOOK_NUM}_real_vs_shuffle_two_step_l1.png"; fig.savefig(fig5_path,dpi=180,bbox_inches="tight"); plt.show(); fig5_path

In [ ]:
# Figure 6 — real vs shuffle L2 residual baseline
fig, ax = plt.subplots(figsize=(8,5)); ax.hist(shuffle_df["shuffle_l2"], bins=25, alpha=0.7, label="shuffle L2"); ax.axvline(higher_order_l2, linewidth=2, label="real L2"); ax.set_title("Real two-step residual L2 vs shuffle baseline"); ax.set_xlabel("two-step residual L2"); ax.set_ylabel("frequency"); ax.legend(); ax.grid(True,alpha=0.3)
fig6_path=FIG_DIR/f"{NOTEBOOK_NUM}_real_vs_shuffle_two_step_l2.png"; fig.savefig(fig6_path,dpi=180,bbox_inches="tight"); plt.show(); fig6_path

In [ ]:
# Figure 7 — top conditional chain residuals
top_chain=conditional_chain_df.head(TOP_N).iloc[::-1]
fig, ax = plt.subplots(figsize=(10,7)); ax.barh(top_chain["chain"], top_chain["abs_conditional_delta"]); ax.set_title("Top conditional chain residuals"); ax.set_xlabel("absolute delta |P(k|i,j)-P(k|j)|"); ax.set_ylabel("chain"); ax.grid(True,axis="x",alpha=0.3)
fig7_path=FIG_DIR/f"{NOTEBOOK_NUM}_top_conditional_chain_residuals.png"; fig.savefig(fig7_path,dpi=180,bbox_inches="tight"); plt.show(); fig7_path

In [ ]:
# Figure 8 — windowed two-step residual norms
fig, ax = plt.subplots(figsize=(8,5)); ax.plot(window_higher_order_df["midpoint"], window_higher_order_df["two_step_l1"], marker="o", label="L1 residual"); ax.plot(window_higher_order_df["midpoint"], window_higher_order_df["two_step_l2"], marker="o", label="L2 residual"); ax.set_xscale("log"); ax.set_title("Windowed two-step residual norms"); ax.set_xlabel("window midpoint x"); ax.set_ylabel("residual norm"); ax.legend(); ax.grid(True,alpha=0.3)
fig8_path=FIG_DIR/f"{NOTEBOOK_NUM}_windowed_two_step_residual_norms.png"; fig.savefig(fig8_path,dpi=180,bbox_inches="tight"); plt.show(); fig8_path

In [ ]:
# Figure 9 — windowed two-step residual heatmap
transition_labels=[f"{a}->{c}" for a in RESIDUES30 for c in RESIDUES30]
pivot=window_delta_grid_df.pivot_table(index="window_index", columns="two_step_transition", values="delta", aggfunc="mean").reindex(columns=transition_labels)
fig, ax = plt.subplots(figsize=(14,6)); im=ax.imshow(pivot.values, aspect="auto", interpolation="nearest", origin="lower"); ax.set_title("Windowed two-step residual delta heatmap"); ax.set_xlabel("two-step transition"); ax.set_ylabel("window index"); ax.set_xticks(np.arange(len(transition_labels))[::4]); ax.set_xticklabels(transition_labels[::4], rotation=90, fontsize=7); fig.colorbar(im, ax=ax, label="Q_window - P_window²")
fig9_path=FIG_DIR/f"{NOTEBOOK_NUM}_windowed_two_step_residual_heatmap.png"; fig.savefig(fig9_path,dpi=180,bbox_inches="tight"); plt.show(); fig9_path

In [ ]:
# Figure 10 — entropy comparison by anchor residue
fig, ax = plt.subplots(figsize=(8,5)); ax.plot(RESIDUE_LABELS,row_entropy_P,marker="o",label="first-order entropy"); ax.plot(RESIDUE_LABELS,row_entropy_Q,marker="o",label="two-step entropy"); ax.axhline(1.0,linestyle="--",label="maximum entropy"); ax.set_title("First-order vs two-step entropy by anchor residue"); ax.set_xlabel("anchor residue mod30"); ax.set_ylabel("normalized entropy"); ax.set_ylim(0,1.05); ax.legend(); ax.grid(True,alpha=0.3)
fig10_path=FIG_DIR/f"{NOTEBOOK_NUM}_entropy_comparison_by_anchor.png"; fig.savefig(fig10_path,dpi=180,bbox_inches="tight"); plt.show(); fig10_path

In [ ]:
# Figure 11 — conditional-memory proxy by middle residue
mi_proxy_rows=[]
for b in RESIDUES30:
    sub=conditional_chain_df[conditional_chain_df["middle_residue"]==b]
    mi_proxy_rows.append({"middle_residue":int(b),"mean_abs_conditional_delta":float(sub["abs_conditional_delta"].mean()) if len(sub) else 0.0,"max_abs_conditional_delta":float(sub["abs_conditional_delta"].max()) if len(sub) else 0.0})
mutual_information_proxy_df=pd.DataFrame(mi_proxy_rows)
fig, ax = plt.subplots(figsize=(8,5)); ax.plot(mutual_information_proxy_df["middle_residue"].astype(str), mutual_information_proxy_df["mean_abs_conditional_delta"], marker="o", label="mean |conditional delta|"); ax.plot(mutual_information_proxy_df["middle_residue"].astype(str), mutual_information_proxy_df["max_abs_conditional_delta"], marker="o", label="max |conditional delta|"); ax.set_title("Conditional-memory proxy by middle residue"); ax.set_xlabel("middle residue mod30"); ax.set_ylabel("conditional-memory proxy"); ax.legend(); ax.grid(True,alpha=0.3)
fig11_path=FIG_DIR/f"{NOTEBOOK_NUM}_conditional_memory_proxy_by_middle_residue.png"; fig.savefig(fig11_path,dpi=180,bbox_inches="tight"); plt.show(); fig11_path

In [ ]:
# Figure 12 — first-order vs two-step lift comparison
next_probs=np.array([np.mean(residues[1:]==r) for r in RESIDUES30]); two_step_probs=np.array([np.mean(r2==r) for r in RESIDUES30])
first_lift=np.divide(P,next_probs.reshape(1,-1),out=np.zeros_like(P),where=next_probs.reshape(1,-1)>0)
two_step_lift=np.divide(Q_emp,two_step_probs.reshape(1,-1),out=np.zeros_like(Q_emp),where=two_step_probs.reshape(1,-1)>0)
lift_delta=two_step_lift-first_lift
fig, ax = plt.subplots(figsize=(9,7)); im=ax.imshow(lift_delta, aspect="auto"); ax.set_title("Two-step lift minus first-order lift"); ax.set_xlabel("target residue mod30"); ax.set_ylabel("anchor residue mod30"); ax.set_xticks(np.arange(n_res)); ax.set_yticks(np.arange(n_res)); ax.set_xticklabels(RESIDUE_LABELS); ax.set_yticklabels(RESIDUE_LABELS); fig.colorbar(im, ax=ax, label="lift delta")
for i in range(n_res):
    for j in range(n_res): ax.text(j,i,f"{lift_delta[i,j]:+.2f}",ha="center",va="center",fontsize=7)
fig12_path=FIG_DIR/f"{NOTEBOOK_NUM}_two_step_minus_first_order_lift_heatmap.png"; fig.savefig(fig12_path,dpi=180,bbox_inches="tight"); plt.show(); fig12_path

## 10. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**

In [ ]:
top_two_step = two_step_delta_df.iloc[0]
top_chain = conditional_chain_df.iloc[0]
measurement = {
    "higher_order_l1": higher_order_l1,
    "higher_order_l2": higher_order_l2,
    "higher_order_max_abs": higher_order_max_abs,
    "mean_first_order_entropy": float(row_entropy_P.mean()),
    "mean_two_step_entropy": float(row_entropy_Q.mean()),
    "mean_entropy_delta": float(row_entropy_delta.mean()),
    "shuffle_mean_l1": shuffle_summary["shuffle_mean_l1"],
    "shuffle_mean_l2": shuffle_summary["shuffle_mean_l2"],
    "l1_excess_over_shuffle": shuffle_summary["l1_excess_over_shuffle"],
    "l2_excess_over_shuffle": shuffle_summary["l2_excess_over_shuffle"],
    "top_two_step_transition": str(top_two_step["two_step_transition"]),
    "top_two_step_abs_delta": float(top_two_step["abs_delta"]),
    "top_conditional_chain": str(top_chain["chain"]),
    "top_conditional_abs_delta": float(top_chain["abs_conditional_delta"]),
    "windowed_two_step_l1_min": float(window_higher_order_df["two_step_l1"].min()),
    "windowed_two_step_l1_max": float(window_higher_order_df["two_step_l1"].max()),
}
cgcs_score = 1.0 / (1.0 + measurement["higher_order_l1"] + max(0.0, measurement["l1_excess_over_shuffle"]) + abs(measurement["mean_entropy_delta"]))
cgcs = {"score": float(cgcs_score), "definition": "1/(1 + higher-order L1 + positive L1 excess over shuffle + |entropy delta|)", "interpretation": "Higher score indicates smaller higher-order residual, smaller excess over shuffle, and stable entropy across order."}
interpretation = f"""
# {NOTEBOOK_TITLE}

## Constraint result

This notebook tests whether prime residue transitions contain higher-order memory beyond a first-order transition operator.

The empirical two-step residual is:

$$
\Delta^{{(2)}} = Q - P^2.
$$

## Remains under constraint

Measured values:

- higher-order L1 = {measurement['higher_order_l1']:.6f}
- higher-order L2 = {measurement['higher_order_l2']:.6f}
- higher-order max absolute residual = {measurement['higher_order_max_abs']:.6f}
- mean first-order entropy = {measurement['mean_first_order_entropy']:.6f}
- mean two-step entropy = {measurement['mean_two_step_entropy']:.6f}

## Drift

Strongest two-step residual transition:

- {measurement['top_two_step_transition']}, |delta| = {measurement['top_two_step_abs_delta']:.6f}

Strongest conditional chain residual:

- {measurement['top_conditional_chain']}, |delta| = {measurement['top_conditional_abs_delta']:.6f}

## Shuffle baseline

- shuffle mean L1 = {measurement['shuffle_mean_l1']:.6f}
- real-minus-shuffle L1 = {measurement['l1_excess_over_shuffle']:.6f}

## CGCS score

$$
CGCS_{{higher}} = {cgcs_score:.6f}.
$$

## Caution

This notebook does not claim prime residues are generated by a second-order Markov process. It uses transition operators as finite measurement tools for local arithmetic memory.
""".strip()
print(interpretation)

## 11. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summaries
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes

In [ ]:
summary_df = pd.DataFrame([{**params, **summary, **measurement, "cgcs_score": cgcs["score"], "cgcs_definition": cgcs["definition"]}])
summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
first_operator_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_operator_P.csv"
two_step_emp_path = DATA_DIR / f"{NOTEBOOK_NUM}_empirical_two_step_operator_Q.csv"
two_step_markov_path = DATA_DIR / f"{NOTEBOOK_NUM}_markov_predicted_two_step_operator_P2.csv"
two_step_delta_path = DATA_DIR / f"{NOTEBOOK_NUM}_two_step_delta_operator.csv"
two_step_delta_records_path = DATA_DIR / f"{NOTEBOOK_NUM}_top_two_step_residual_transitions.csv"
operator_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_operator_metrics.csv"
entropy_comparison_path = DATA_DIR / f"{NOTEBOOK_NUM}_entropy_comparison_by_anchor.csv"
shuffle_path = DATA_DIR / f"{NOTEBOOK_NUM}_shuffle_two_step_residual_baseline.csv"
shuffle_summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_shuffle_residual_summary.csv"
conditional_chain_path = DATA_DIR / f"{NOTEBOOK_NUM}_conditional_two_step_chain_residuals.csv"
window_higher_order_path = DATA_DIR / f"{NOTEBOOK_NUM}_windowed_higher_order_residual_metrics.csv"
window_delta_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_windowed_two_step_residual_grid.csv"
mutual_proxy_path = DATA_DIR / f"{NOTEBOOK_NUM}_conditional_memory_proxy.csv"
lift_delta_path = DATA_DIR / f"{NOTEBOOK_NUM}_two_step_minus_first_order_lift.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"
interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"
summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
P_df.to_csv(first_operator_path)
Q_emp_df.to_csv(two_step_emp_path)
Q_markov_df.to_csv(two_step_markov_path)
Delta2_df.to_csv(two_step_delta_path)
two_step_delta_df.to_csv(two_step_delta_records_path, index=False)
operator_metrics_df.to_csv(operator_metrics_path, index=False)
entropy_comparison_df.to_csv(entropy_comparison_path, index=False)
shuffle_df.to_csv(shuffle_path, index=False)
shuffle_summary_df.to_csv(shuffle_summary_path, index=False)
conditional_chain_df.to_csv(conditional_chain_path, index=False)
window_higher_order_df.to_csv(window_higher_order_path, index=False)
window_delta_grid_df.to_csv(window_delta_grid_path, index=False)
mutual_information_proxy_df.to_csv(mutual_proxy_path, index=False)
pd.DataFrame(lift_delta, index=RESIDUE_LABELS, columns=RESIDUE_LABELS).to_csv(lift_delta_path)

figure_paths = [fig1_path,fig2_path,fig3_path,fig4_path,fig5_path,fig6_path,fig7_path,fig8_path,fig9_path,fig10_path,fig11_path,fig12_path]
metadata = {"params":params,"summary":summary,"measurement":measurement,"shuffle_summary":shuffle_summary,"cgcs":cgcs,"figures":[str(p) for p in figure_paths]}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
figures_md = "

## Figures

"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_',' ').title()}

![Figure {i}](../figures/{fig.name})

"
interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")
design_notes_md_path.write_text(f"# Design Notes — {NOTEBOOK_TITLE}

Notebook 17 follows Notebook 16 by testing whether residue transition structure contains higher-order memory beyond the first-order operator.

## Constraint

$$\Delta^{{(2)}} = Q - P^2.$$

## Handoff

Notebook 18 should test residue-memory compression, predictive recoverability, or null-model families beyond simple shuffling.
", encoding="utf-8")
summary_tex_path.write_text(f"\section*{{{NOTEBOOK_TITLE}}}

This notebook tests higher-order residue transition memory beyond a first-order transition operator.\n
Higher-order L1: {measurement['higher_order_l1']:.6f}. CGCS higher-order score: {cgcs_score:.6f}.
", encoding="utf-8")
math_tex_path.write_text(f"\documentclass{{article}}
\usepackage{{amsmath}}
\begin{{document}}
\section*{{Math Notes: {NOTEBOOK_TITLE}}}
\[P_{{ij}}=\Pr(r_{{n+1}}=j\mid r_n=i)\]
\[Q_{{ik}}=\Pr(r_{{n+2}}=k\mid r_n=i)\]
\[\Delta^{{(2)}}=Q-P^2\]
\end{{document}}
", encoding="utf-8")
summary_path, first_operator_path, two_step_emp_path, two_step_markov_path, two_step_delta_path, operator_metrics_path, shuffle_path, conditional_chain_path, window_higher_order_path, metadata_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 12. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 13. Next notebook handoff

Next:

> Notebook 18 should test residue-memory compression, predictive recoverability, or null-model families beyond simple shuffling.

In [ ]:
next_step = "Notebook 18: residue-memory compression, predictive recoverability, or expanded null-model families."
print(next_step)